<a href="https://colab.research.google.com/github/sanil-edwin/llm-agents/blob/main/L10_agents_cheese_quiz_guild.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="padding-top: 25px;padding-bottom: 25px;text-align: left; padding-left: 10px; background-color: #DDDDDD;
    color: black;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/Harvard-IACS/2018-CS109A/master/content/styles/iacs.png" height="50px"> <a href='https://harvard-iacs.github.io/2025-AC215/' target='_blank'><strong><font color="#A41034">AC215/E115: MLOps & LLMOps: Production AI Systems</font></strong></a></h1>

# **<font color="#A41034">Tutorial -  Agents - 🧀 The Cheesy Way 🧀</font>**

**Harvard University**<br/>
**Fall 2025**<br/>
**Instructor:** Pavlos Protopapas<br/>


<hr style="height:2pt">

## 📝 Make a Copy to Edit

This notebook is **view-only**. To edit it, follow these steps:

1. Click **File** > **Save a copy in Drive**.
2. Your own editable copy will open in a new tab.

Now you can modify and run the code freely!

# **Working with Agents**


<center><img src="https://drive.google.com/uc?export=view&id=1oCldVicFepwdYbPHcD8Ui4VeyQVJYgut" height="400"><center>



To understand how agents work, we will start from a basic LLM and work our way all the way to complex multi agent system. Each step will introduce us to more concepts we learnt in lecture and add features so that we get a solid understanding.

We will use the [OpenAI Python library](https://github.com/openai/openai-python?tab=readme-ov-file) that provides convenient access to the OpenAI REST API to all their foundation models.

We will build agents using [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/). The OpenAI Agents SDK enables you to build agentic AI apps in a lightweight, easy-to-use package with very few abstractions.


# **Learning Objectives**

By the end of this tutorial/guided demo, you will be able to:

- **Understand AI Agents:**
  - Compare a LLM, Reasoning LLM, and Agent.
  - Define agents and thier building blocks.

- **Context Engineering:**
  - Understand the various components of Context Engineering

- **Tools & MCP (Model Context Protocol):**
  - Build a simple tool and make an agent use it.
  - Use prebuilt MCP tools or build your own.

- **Multi Agent Systems:**
  - Breaking a task into smaller micro agent tasks.
  - Orchestrating a multi agent system that builds a cheese quiz


# **Understanding Agents**

**What is an Agent?**<br>
---

Agent is a component that uses an LLM with instructions and tools, that can run autonomously to accomplish tasks.


# Prerequisites

Let's make sure we have everything needed. We'll need to have an OpenAI API key for some parts of this tutorial.

Before we can start using the [OpenAI API](https://openai.com/blog/openai-api), we'll need to sign up for an API key from OpenAI. We can do this by visiting the [OpenAI API Keys](https://platform.openai.com/api-keys) page and creating a new API key.

In [ ]:
# @title Install Packages
# Installing necessary libraries
!pip -q install fastmcp openai-agents > /dev/null 2>&1
#%pip install openai-agents > /dev/null 2>&1

In [ ]:
# @title Imports
import os
import asyncio
import html
import json, random, time, threading
import openai
from openai import OpenAI
from pathlib import Path
from typing import Dict, Any
from dataclasses import dataclass
from typing import List, Optional, Dict, Literal
from jsonschema import validate as js_validate, ValidationError
from fastmcp import FastMCP
from contextlib import AsyncExitStack
from pydantic import BaseModel, Field, ValidationError
from agents import (
    Agent,
    Runner,
    WebSearchTool,
    function_tool,
    SQLiteSession,
    ModelSettings,
    FunctionTool
)
from agents.agent_output import AgentOutputSchemaBase
from agents.mcp import MCPServerStreamableHttp
from agents.exceptions import ModelBehaviorError
from agents.items import ToolCallItem, ToolCallOutputItem

from IPython.display import HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("openai.agents").setLevel(logging.ERROR)

In [ ]:
# @title Setup OpenAI Key
# Using Google Colab's Secrets feature 🔑
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
import os
os.environ['OPENAI_API_KEY'] = api_key

#Proper client initialization
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

In [ ]:
# @title Download
# Download files
!gdown 1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh # https://drive.google.com/file/d/1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh/view?usp=drive_link
!gdown 1OykEQrwa5cnZgPIVrOdnDeq3txots3H0 # https://drive.google.com/file/d/1OykEQrwa5cnZgPIVrOdnDeq3txots3H0/view?usp=drive_link

Downloading...
From: https://drive.google.com/uc?id=1dxvxGZZ-yYEEHQuaoP4HwaYxsbOC4XEh
To: /content/quiz-schema.json
100% 2.10k/2.10k [00:00<00:00, 5.60MB/s]
Downloading...
From: https://drive.google.com/uc?id=1OykEQrwa5cnZgPIVrOdnDeq3txots3H0
To: /content/quiz-ui.html
100% 10.8k/10.8k [00:00<00:00, 68.2MB/s]


---

# **LLM to Agent**

To understand how an Agent works let us use a sample cheese question: **What's the best cheese for a grilled cheese sandwich?**

We will ask this same question to a a base LLM, Reasoning LLM, and an Agent.

Compare the responses in each step to understand the concepts.

In [ ]:
# Define the LLM model and question
model = "gpt-4o"
user_prompt = "What's the best cheese for a grilled cheese sandwich?"

## **Base LLM**

First we ask the question to a LLM direclty with no other bells and whistles.

In [ ]:
# @title Define BaseLLM
class BaseLLM:
    """Demonstrates basic LLM functionality - single call, direct response"""

    def __init__(self, model="gpt-4o"):
        self.model = model

    def generate(self, user_prompt: str,max_tokens=1000):
        """Single forward pass through LLM"""
        print("🧠 BaseLLM: Making single API call...")
        start_time = time.time()
        response = client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": user_prompt}],
            max_tokens=max_tokens
        )
        end_time = time.time()

        return {
          "content": response.choices[0].message.content,
          "tokens_used": response.usage.total_tokens,
          "response_time": end_time - start_time
      }

In [ ]:
# Initialize
base_llm = BaseLLM(model)
llm_result = base_llm.generate(user_prompt)

print("\n" + "="*50)
print("BASE LLM:",model)
print("="*50)
print(f"Response:")
display(Markdown(llm_result['content']))
print(f"Tokens: {llm_result['tokens_used']}, Time: {llm_result['response_time']:.2f}s")

🧠 BaseLLM: Making single API call...

BASE LLM: gpt-4o
Response:


The best cheese for a grilled cheese sandwich can depend on personal taste preferences, but some popular choices include:

1. **American Cheese**: Known for its creamy texture and excellent melting properties, American cheese is a classic choice for grilled cheese sandwiches.

2. **Cheddar**: Both sharp and mild cheddar work well, offering a strong flavor and good meltability. Sharp cheddar provides a more pronounced taste.

3. **Gouda**: This cheese melts beautifully and has a mild, slightly sweet flavor, making it a great option for grilled cheese.

4. **Swiss**: Known for its nutty flavor, Swiss cheese melts well and adds a different dimension to the sandwich.

5. **Monterey Jack**: This cheese melts easily and has a mild flavor, making it a versatile choice that pairs well with other ingredients.

6. **Provolone**: With its smooth texture and mild, tangy flavor, provolone melts nicely and can add interest to your grilled cheese.

You can also blend different cheeses to create a more complex flavor profile. For instance, combining cheddar and mozzarella can give you a mix of strong taste and optimal meltability. Ultimately, the best cheese is one that delivers the flavor and texture you enjoy most.

Tokens: 268, Time: 8.97s


## **Reasoning LLM**

<img src="https://drive.google.com/uc?export=view&id=1Us_IdQPDZVH8I6ZErXzENItd94KrfNS8" height="400">


Next we build a Reasoning LLM that does the following:
* Step 1: Analyze the type of query and create reasoning plan
* Step 2: Break down into specific sub-questions
* Step 3: Answer each sub-question specifically
* Step 4: Verify accuracy and completeness
* Step 5: Combine all verified answers into structured response
* Orchestrate Step 1 - 5 to get the final answer

In [ ]:
# @title Define ReasoningLLM
class ReasoningLLM:
    """Demonstrates reasoning framework - multiple orchestrated calls"""

    def __init__(self, base_llm: BaseLLM):
        self.base_llm = base_llm
        self.conversation_history = []

    def analyze_query(self, query: str) -> Dict[str, Any]:
        """Step 1: Analyze the type of query and create reasoning plan"""
        analysis_prompt = f"""
        Analyze this query and determine the best approach to answer it comprehensively:
        Query: "{query}"

        Respond with:
        1. Query type (factual, process, comparison, etc.)
        2. Key components to address
        3. Suggested breakdown steps

        Keep response concise and structured.
        """

        print("🔧 ReasoningLLM: Step 1 - Analyzing query...")
        response = self.base_llm.generate(analysis_prompt)
        return {"analysis": response["content"], "tokens": response["tokens_used"]}

    def decompose_problem(self, query: str, analysis: str) -> List[str]:
        """Step 2: Break down into specific sub-questions"""
        decomposition_prompt = f"""
        Based on this analysis: {analysis}

        Break down the query "{query}" into 3-4 specific sub-questions that, when answered together,
        will provide a comprehensive response. List only the questions, one per line.
        """

        print("🔧 ReasoningLLM: Step 2 - Decomposing problem...")
        response = self.base_llm.generate(decomposition_prompt)

        # Extract questions from response
        questions = [q.strip() for q in response['content'].split('\n') if q.strip() and '?' in q]
        return questions[:4]  # Limit to 4 questions

    def answer_sub_question(self, question: str) -> str:
        """Step 3: Answer each sub-question specifically"""
        focused_prompt = f"""
        Answer this specific question about cheese making with precise, technical details:
        {question}

        Provide a clear, factual answer focusing only on this aspect.
        """

        print(f"🔧 ReasoningLLM: Step 3 - Answering: {question[:50]}...")
        response = self.base_llm.generate(focused_prompt)
        return response["content"]

    def verify_response(self, question: str, answer: str) -> Dict[str, Any]:
        """Step 4: Verify accuracy and completeness"""
        verification_prompt = f"""
        Question: {question}
        Answer: {answer}

        Evaluate this answer on a scale of 1-10 for:
        1. Accuracy
        2. Completeness
        3. Clarity

        Respond with just three numbers and any critical missing information.
        """

        print("🔧 ReasoningLLM: Step 4 - Verifying response...")
        response = self.base_llm.generate(verification_prompt)
        return {"verification": response["content"]}

    def synthesize_final_response(self, query: str, qa_pairs: List[tuple]) -> str:
        """Step 5: Combine all verified answers into structured response"""
        synthesis_prompt = f"""
        Original question: {query}

        Sub-questions and answers with verifications:
        {chr(10).join([
            f"Q: {qa['question']}\nA: {qa['answer']}\nVerification: {qa['verification']}\n"
            for qa in qa_pairs
        ])}

        Synthesize these into a well-structured, comprehensive answer to the original question.
        Use clear steps/sections and ensure logical flow.
        """

        print("🔧 ReasoningLLM: Step 5 - Synthesizing final response...")
        response = self.base_llm.generate(synthesis_prompt)
        return response["content"]

    def reason(self, query: str) -> Dict[str, Any]:
        """Main reasoning orchestration method"""
        print(f"\n🔧 ReasoningLLM: Processing '{query}'")
        start_time = time.time()

        # Step 1: Analyze query
        analysis = self.analyze_query(query)

        # Step 2: Decompose into sub-questions
        sub_questions = self.decompose_problem(query, analysis['analysis'])

        # Step 3 & 4: Answer and verify each sub-question
        qa_pairs = []
        total_tokens = analysis['tokens']

        for question in sub_questions:
            answer = self.answer_sub_question(question)
            verification = self.verify_response(question, answer)
            qa_pairs.append({
                "question": question,
                "answer": answer,
                "verification": verification["verification"]
            })
            total_tokens += 100  # Approximate token usage

        # Step 5: Synthesize final response
        final_response = self.synthesize_final_response(query, qa_pairs)

        end_time = time.time()

        return {
            "response": final_response,
            "reasoning_steps": {
                "analysis": analysis['analysis'],
                "sub_questions": sub_questions,
                "qa_pairs": qa_pairs
            },
            "total_tokens": total_tokens,
            "processing_time": end_time - start_time
        }

In [ ]:
# Initialize
base_llm = BaseLLM(model)
reasoning_llm = ReasoningLLM(base_llm)

print("\n" + "="*50)
print("REASONING LLM:",model)
print("="*50)
reasoning_result = reasoning_llm.reason(user_prompt)
print(f"Final Response:")
display(Markdown(reasoning_result['response']))
print(f"Sub-questions explored: {len(reasoning_result['reasoning_steps']['sub_questions'])}")
print(f"Total tokens: {reasoning_result['total_tokens']}, Time: {reasoning_result['processing_time']:.2f}s")


REASONING LLM: gpt-4o

🔧 ReasoningLLM: Processing 'What's the best cheese for a grilled cheese sandwich?'
🔧 ReasoningLLM: Step 1 - Analyzing query...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 2 - Decomposing problem...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 1. What are the key criteria that make a cheese su...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 4 - Verifying response...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 2. What are some popular cheeses commonly used in ...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 4 - Verifying response...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 3. What are some variations or alternatives in che...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 4 - Verifying response...
🧠 BaseLLM: Making single API call...
🔧 ReasoningLLM: Step 3 - Answering: 4. Based on the criteria and options available, wh...
🧠 Ba

When selecting the best cheese for a grilled cheese sandwich, several key factors should be considered to ensure a delightful experience. Here’s a comprehensive approach that synthesizes the criteria, popular cheese options, variations for dietary needs, and practical selection strategies:

### Key Criteria for Cheese Selection

1. **Meltability**: A critical trait for grilled cheese sandwiches, meltability ensures the cheese melts smoothly and uniformly. Cheeses with higher moisture and fat content, such as Mozzarella, American, and Monterey Jack, excel in this aspect.

2. **Flavor Profile**: The flavor should complement rather than overpower the sandwich. Ideally, the cheese should offer a balance of sharpness, richness, and creaminess. Cheddar offers a tangy flavor, while Gouda and Gruyère add a nutty, sweet complexity.

3. **Texture**: The desired texture should allow for easy slicing and provide a creamy consistency when melted. Semi-soft cheeses tend to achieve this balance effectively.

### Popular Cheese Options

- **Cheddar**: Known for its gooey melt and rich, sharp flavor, cheddar is a classic choice.
- **American Cheese**: Offers a smooth, creamy melt with a mild flavor, ideal for a classic experience.
- **Mozzarella**: Its high moisture content results in a stretchy, gooey melt with a mild, milky taste.
- **Swiss Cheese**: Emmental and Gruyère bring a creamy texture accompanied by a nutty, slightly sweet flavor.
- **Provolone**: Provides a smooth melt and a mild, slightly smoky taste.

### Considerations for Dietary Preferences

1. **Lactose-Free Options**: Cheeses like aged Cheddar and Swiss naturally have low lactose, while specific brands offer lactose-free choices.
2. **Vegan and Non-Dairy Alternatives**: Plant-based cheeses made from nuts or soy mimic the melt and flavor of dairy cheeses while being suitable for vegans.
3. **Gourmet Choices**: Artisan cheeses crafted with unique flavors can elevate the taste, as can those infused with herbs or spices.
4. **Low-Fat and Reduced-Sodium Options**: For health-conscious consumers, modified dairy cheeses retain flavor while reducing fat or sodium.

### Recommended Approach for Selection

- **Single Cheese**: Use a medium to sharp Cheddar for its classic appeal, or opt for Mozzarella or Havarti if meltability is the primary consideration.
- **Combination**: Blending cheeses can enhance both flavor and texture. Pairing sharp Cheddar with a milder Havarti balances strong flavors with creaminess, while combining Gruyère with Cheddar offers a robust, nutty twist.
- **Experimentation**: Consider incorporating flavored cheeses, such as smoked Gouda, to add depth and uniqueness to your sandwich.

Ultimately, the best cheese for a grilled cheese sandwich depends on personal taste preferences while ensuring the necessary melting properties and complementary flavors are in place for a satisfying culinary experience.

Sub-questions explored: 4
Total tokens: 643, Time: 76.53s


## **Agent**

Agent is a component that uses an LLM with instructions and tools, that can run autonomously to accomplish tasks.

<center><img src="https://drive.google.com/uc?export=view&id=1TgpuVStg8ChZBMubgd-_Ty2wrMdGWcOY" height="400"><center>

Next we build an Agent that does the following:
* Perform the same reasoning stratedgy as our reasoning LLM
* Add a web search tool so the agent has capabilty to fact check using the internet

In [ ]:
# @title Define SimpleAgent
class SimpleAgent:
    """Demonstrates agent functionality - LLM, instructions, tool usage (research-oriented)"""

    def __init__(
        self,
        model: str = "gpt-4o",
        user_location: Dict[str, Any] | None = None,
        search_context_size: str = "medium",          # "low" | "medium" | "high"
        instructions: str | None = None,
    ):
        # Configure the web search tool
        web_tool = WebSearchTool(
            user_location=user_location,
            search_context_size=search_context_size
        )

        # Instructions similar to Reasoning LLM
        if instructions is None:
            instructions = (
                "You are a meticulous research agent. "
                "Before answering, outline a brief plan with 3–4 sub-questions. "
                "Use the web_search tool for up-to-date or factual details. "
                "Perform at least TWO web_search calls across DIFFERENT reputable domains. "
                "Cross-check key facts (dates, figures, definitions). "
                "Then write a clear, structured answer with sections and practical recommendations. "
                "Finish with a 'Sources' list (title + URL) covering the items you used."
            )

        self.agent = Agent(
            name="SimpleAgent",
            instructions=instructions,
            model=model,
            tools=[web_tool],
        )

    def _research_prompt(self, query: str) -> str:
        return f"""
Task: Answer the user's question comprehensively.

User question: {query}

Workflow you must follow:
1) **Plan**: List 3–4 concise sub-questions you will answer.
2) **Research**: Use `web_search` at least twice across different domains to gather facts.
3) **Synthesize**: Combine findings into a cohesive answer that is better than a generic LLM response.
4) **Verify**: Re-check critical claims and resolve conflicts if sources disagree.
5) **Present**: Write the final answer with these sections:
   - Summary (2–4 sentences)
   - Key Criteria / How to Choose
   - Top Recommendations (with brief justifications)
   - Smart Variations or Blends (if applicable)
   - Practical Tips (actionable steps)
   - Sources (bullet list: Title — URL)

Output format: Markdown.
If uncertainty remains, say what is uncertain and why.
"""

    async def generate(self, query: str) -> Dict[str, Any]:
        print(f"\n🤖 SimpleAgent: Running with web_search for '{query}'")
        start = time.time()

        # 🔧 pass the guided research prompt
        result = await Runner.run(self.agent, input=self._research_prompt(query))

        end = time.time()
        return {
            "response": result.final_output,
            "processing_time": end - start,
            "tools": self.agent.tools  # available tools (actual usage will show in logs/tracing if enabled)
        }


In [ ]:
# Initialize
agent = SimpleAgent(model)

print("\n" + "="*50)
print("AGENT:",model)
print("="*50)
agent_result = await agent.generate(user_prompt)
print(f"Final Response:")
display(Markdown(agent_result['response']))
print(f"Tools used: {len(agent_result['tools'])}")
print(f"Time: {agent_result['processing_time']:.2f}s")


AGENT: gpt-4o

🤖 SimpleAgent: Running with web_search for 'What's the best cheese for a grilled cheese sandwich?'
Final Response:


**Plan**

Sub‑questions to answer:
1. What key criteria make a cheese ideal for grilled cheese sandwiches (e.g., meltability, flavor, texture)?
2. Which cheeses are most recommended by culinary experts for grilled cheese?
3. What are smart cheese blends or variations to enhance flavor and texture?
4. What practical steps improve the grilled cheese experience (preparation, cooking tips)?

---

## Summary

The best cheese for grilled cheese combines creamy meltability, balanced flavor, and smooth texture. Classic choices include American, Cheddar, Gruyère, Havarti, and Fontina. Blending cheeses (e.g., American + Cheddar or Mozzarella + Pepper Jack) often yields superior results. Use freshly grated cheese, moderate heat, and thoughtful fat and bread choices to perfect your sandwich.

---

## Key Criteria / How to Choose

- **Meltability & Texture**: Look for cheeses that melt smoothly, stretch gracefully, and coat bread evenly without becoming greasy. ([flavor365.com](https://flavor365.com/the-ultimate-guide-to-cheeses-for-a-perfect-grilled-cheese/?utm_source=openai))
- **Flavor Profile**: Mild cheeses like American or Monterey Jack allow buttery bread to shine, while sharper options like aged Cheddar or nutty Gruyère add depth. ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))
- **Chemical Traits**: Cheeses with a pH between 5.3–5.5 (e.g., Gouda, Gruyère) melt particularly well, avoiding clumping. ([time.com](https://time.com/4101512/best-cheese-for-grilled-cheese-sandwiches/?utm_source=openai))
- **Avoid Pre‑shredded**: Anti-caking agents in pre-shredded cheese can lead to grainy melts—grate from a block for better results. ([flavor365.com](https://flavor365.com/the-ultimate-guide-to-cheeses-for-a-perfect-grilled-cheese/?utm_source=openai))

---

## Top Recommendations

- **American Cheese**: Iconic classic—ultra-creamy and perfectly meltable. Nostalgic comfort food staple. ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))
- **Cheddar (mild or sharp)**: Balances flavor and melt; provides satisfying stretch. ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))
- **Gruyère**: Nutty, sweet, and sophisticated with exceptional smooth, stretchy melt. ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))
- **Havarti**: Buttery and silky, a great processed alternative that melts beautifully; offers flavor variants (e.g., dill, jalapeño). ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))
- **Fontina**: Rich, creamy, velvety – unmatched for gooey consistency. ([flavor365.com](https://flavor365.com/9-best-grilled-cheese-cheeses-ultimate-melting-guide/?utm_source=openai))
- **Monterey Jack**: Mild, buttery, smooth melt; excellent base or blending cheese. ([cheesescientist.com](https://cheesescientist.com/lifestyle/best-cheeses-for-a-grilled-cheese-sandwich/?utm_source=openai))

---

## Smart Variations or Blends

- **American + Cheddar**: Classic pairing—Creamy melt from American, sharp kick from Cheddar. Endorsed by Martha Stewart. ([simplyrecipes.com](https://www.simplyrecipes.com/martha-stewart-grilled-cheese-sandwich-review-11715761?utm_source=openai))
- **Mozzarella + Pepper Jack**: Stretchy, gooey base meets spicy, flavorful kick. Recommended by chefs for texture & taste balance. ([cozymeal.com](https://www.cozymeal.com/magazine/best-cheese-for-grilled-cheese?utm_source=openai))
- **Provolone + Blue or Goat**: Soft base with bold, pungent accent for complexity. Top Chef’s greenspan advocates layering for texture and depth. ([bravotv.com](https://www.bravotv.com/top-chef/blogs/top-chefs-reveal-tips-to-making-best-grilled-cheese-sandwich?utm_source=openai))

---

## Practical Tips

1. **Grate your cheese fresh** for optimal melt. ([flavor365.com](https://flavor365.com/the-ultimate-guide-to-cheeses-for-a-perfect-grilled-cheese/?utm_source=openai))  
2. **Use medium heat**, apply butter or mayo on bread for even browning and crunch (mayo gives ultra-crispy result). ([simplyrecipes.com](https://www.simplyrecipes.com/martha-stewart-grilled-cheese-sandwich-review-11715761?utm_source=openai))  
3. **Blend cheeses** thoughtfully: base for melt, accent for flavor. ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))  
4. **Pick sturdy yet complementary bread** (e.g., sourdough, whole grain) that holds up to melting cheese. ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))  
5. **Optional add-ins**: Garlic rub, caramelized onions, chutney, or jam to elevate flavor. ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))

---

## Sources

- We Asked 7 Chefs the Best Cheese to Use for Grilled Cheese — Here’s What They Said — EatingWell ([eatingwell.com](https://www.eatingwell.com/best-cheese-for-grilled-cheese-11821821?utm_source=openai))  
- What's the Best Cheese for Grilled Cheese? (2025 Guide) — Flavor365 ([flavor365.com](https://flavor365.com/the-ultimate-guide-to-cheeses-for-a-perfect-grilled-cheese/?utm_source=openai))  
- The 6 Best Cheeses for Grilled Cheese — Food & Wine ([foodandwine.com](https://www.foodandwine.com/appetizers/antipasto/cheese/best-cheese-for-grilled-cheese?utm_source=openai))  
- 10 Best Cheeses for Grilled Cheese Sandwiches — Parade ([parade.com](https://parade.com/1336503/elizabethnarins/best-cheeses-for-grilled-cheese//?utm_source=openai))  
- 10 Best Cheeses for Grilled Cheese Sandwiches (CheeseScientist) ([cheesescientist.com](https://cheesescientist.com/lifestyle/best-cheeses-for-a-grilled-cheese-sandwich/?utm_source=openai))  
- The 9 Best Cheeses for Grilled Cheese Sandwiches — Flavor365 (melting guide) ([flavor365.com](https://flavor365.com/9-best-grilled-cheese-cheeses-ultimate-melting-guide/?utm_source=openai))  
- What Cheese is Best For Grilled Cheese Sandwiches? — U.S. Dairy ([usdairy.com](https://www.usdairy.com/news-articles/best-cheese-for-grilled-cheese?utm_source=openai))  
- Martha Stewart’s Simple Trick for the Best Grilled Cheese — SimplyRecipes ([simplyrecipes.com](https://www.simplyrecipes.com/martha-stewart-grilled-cheese-sandwich-review-11715761?utm_source=openai))  
- These Are the 3 Best Cheeses for a Grilled Cheese Sandwich According to Science — Time / American Chemical Society ([time.com](https://time.com/4101512/best-cheese-for-grilled-cheese-sandwiches/?utm_source=openai))  

If any uncertainties remain (e.g., regional cheese availability), let me know—I can refine recommendations based on your location or tastes.

Tools used: 1
Time: 16.41s


---

# **Context Engineering**

<center><img src="https://drive.google.com/uc?export=view&id=1ihGnZhckcT_I6oyMJsVGT7ZeofkyxLpD" height="400"><center>




LLMs are stateless functions. To get the best results, you need to give LLMs the best inputs. Context engineering is about building:
* The instructions you give to the model. The system prompt & user prompt
* Any documents or external data you retrieve (e.g. RAG)
* Any tool call output
* Any past state or other history (Short-term Memory)
* Any past messages or events from related but separate histories/conversations (Long-term Memory)
* Instructions about what sorts of structured data to output

***Think of context engineering as the invisible layer that feeds and connects all parts of the agent.***

It manages what information the LLM sees, when it sees it, and how it uses it.
Without context engineering, even the best reasoning model cannot plan, adapt, or remember effectively.

To understand how Context Engineering works let us use a sample cheese question: **What's the best cheese for a grilled cheese sandwich?**

We will ask this same question to an LLM but incremently adjustting the context and compare the results from each step.
* Step 1: Basic prompt
* Step 2: Prompt Engineering
* Step 3: Retrieve Relevant Documents (e.g. RAG)
* Step 4: Adding Short-term Memory
* Step 5: Adding Long-term Memory
* Step 6: Tool Call Outputs
* Step 7: Structured Output

In [ ]:
model = "gpt-4o"
user_prompt = "What's the best cheese for a grilled cheese sandwich?"

In [ ]:
# @title Step 1: Basic prompt
print("\n" + "="*70)
print("Step 1: Basic Prompt (No Context Engineering)")
print("="*70)

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 1: Basic Prompt (No Context Engineering)


The best cheese for a grilled cheese sandwich often depends on personal preference, but some popular choices include:

1. **Cheddar**: A classic choice, cheddar melts well and offers a sharp, tangy flavor that complements the buttery bread.

2. **American**: Known for its excellent melting properties, American cheese provides a creamy texture and mild flavor, making it a favorite for many.

3. **Gruyère**: This Swiss cheese adds a nutty, slightly sweet flavor and melts beautifully, making it a great option for a more gourmet grilled cheese.

4. **Monterey Jack**: With a mild flavor and good melting qualities, Monterey Jack is another excellent choice, often paired with other cheeses for added depth.

5. **Mozzarella**: Known for its stretchiness and mild taste, mozzarella can be a great addition, especially when combined with more flavorful cheeses.

6. **Provolone**: Offering a mild, slightly tangy flavor, provolone melts well and can add a nice depth to your sandwich.

For the best results, you might consider using a combination of cheeses to balance flavor and texture. For example, pairing a sharp cheddar with a creamy mozzarella can create a deliciously gooey and flavorful sandwich.

In [ ]:
# @title Step 2: Prompt Engineering
print("\n" + "="*70)
print("Step 2: With Prompt Engineering")
print("="*70)

# Well-crafted system prompt
system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning."""

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 2: With Prompt Engineering


When it comes to crafting the perfect grilled cheese sandwich, the choice of cheese is crucial for achieving that gooey, melty interior and a balanced flavor profile. Here are some top contenders:

1. **American Cheese**: This is a classic choice for grilled cheese sandwiches due to its excellent melting properties. It melts smoothly at relatively low temperatures, creating a creamy texture. Its mild flavor is familiar and comforting, making it a favorite for many.

2. **Cheddar**: A sharp or medium cheddar is another popular option. It offers a more robust flavor compared to American cheese. Cheddar melts well, especially when grated, and provides a nice balance of sharpness and creaminess. For optimal melting, aim for a temperature around 150°F (65°C).

3. **Gruyère**: Known for its nutty and slightly sweet flavor, Gruyère melts beautifully, making it an excellent choice for a more gourmet grilled cheese. It melts at around 130°F (54°C) and pairs well with other cheeses like cheddar for a complex flavor profile.

4. **Fontina**: This cheese is semi-soft and has a buttery, slightly nutty flavor. It melts smoothly and evenly, making it a great choice for a rich and creamy grilled cheese. Fontina melts at about 150°F (65°C).

5. **Mozzarella**: While not as flavorful on its own, mozzarella provides a stretchy, gooey texture that many people love. It melts at around 130°F (54°C) and can be combined with a more flavorful cheese like cheddar or Gruyère for added depth.

For the best results, consider using a combination of cheeses to balance flavor and texture. For instance, mixing cheddar with a bit of Gruyère or mozzarella can give you both the sharpness and the meltability you desire. Additionally, always grate your cheese for even melting and use a moderate heat to avoid burning the bread before the cheese has fully melted.

In [ ]:
# @title Step 3: Retrieve Relevant Documents (e.g. RAG)
# Simulated knowledge base (in real apps, this would come from a vector database)
CHEESE_KNOWLEDGE_BASE = {
    "american_cheese": {
        "melting_temp": "150°F",
        "properties": "Smooth melt, mild flavor, contains emulsifiers for perfect texture",
        "best_for": "Classic grilled cheese, kids' sandwiches"
    },
    "cheddar": {
        "melting_temp": "170°F",
        "properties": "Rich flavor, can separate if overheated, varies by age",
        "best_for": "Flavor-focused sandwiches, adults"
    },
    "gruyere": {
        "melting_temp": "170°F",
        "properties": "Nutty and complex, excellent melt, slightly sweet",
        "best_for": "Gourmet grilled cheese, French onion soup"
    }
}

print("\n" + "="*70)
print("Step 3: With RAG (Retrieved Knowledge)")
print("="*70)

# System prompt
system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning."""

# Inject retrieved knowledge into the prompt
enhanced_user_prompt = f"""Based on this cheese data:
{str(CHEESE_KNOWLEDGE_BASE)}
User question: {user_prompt}"""

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": enhanced_user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 3: With RAG (Retrieved Knowledge)


For a grilled cheese sandwich, the choice of cheese depends on the flavor profile and texture you desire. Here's a breakdown based on the data provided:

1. **American Cheese**: This is an excellent choice for a classic grilled cheese sandwich. It melts smoothly at a relatively low temperature of 150°F, which makes it easy to achieve that gooey, stretchy texture that many people love in a grilled cheese. Its mild flavor is universally appealing, especially for kids, and the emulsifiers ensure a consistent melt without separation.

2. **Cheddar**: If you're looking for a more flavor-focused sandwich, cheddar is a great option. It has a richer taste that can range from mild to sharp, depending on its age. However, it melts at a slightly higher temperature of 170°F and can separate if overheated, so it's important to cook it gently. Cheddar is ideal for those who want a more robust flavor in their grilled cheese.

3. **Gruyere**: For a gourmet twist, Gruyere is an excellent choice. It melts beautifully at 170°F and offers a nutty, complex flavor with a hint of sweetness. This cheese is perfect if you're looking to elevate your grilled cheese to something more sophisticated, perhaps paired with additional ingredients like caramelized onions or mushrooms.

**Recommendation**: If you're aiming for a classic, kid-friendly grilled cheese, go with American cheese for its smooth melt and mild flavor. For a more adult, flavor-rich sandwich, cheddar is a great choice, but be mindful of the cooking temperature to prevent separation. For a gourmet experience, Gruyere will provide a delightful complexity and excellent melt. You can also consider blending these cheeses to balance flavor and texture, such as combining cheddar with American for a rich yet smooth melt.

In [ ]:
# @title Step 4: Adding Short-term Memory
print("\n" + "="*70)
print("Step 4: With Conversation State/History (Short-term Memory)")
print("="*70)

system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

Remember the user's preferences and previous questions in this conversation"""

# Build conversation history
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm cooking for my kids who prefer mild flavors"},
    {"role": "assistant", "content": "Great! I'll keep that in mind. Kids often prefer milder, creamier cheeses. What would you like to make?"},
    {"role": "user", "content": "Should I use butter or mayo on the bread?"},
    {"role": "assistant", "content": "For kids, I'd recommend butter on the outside of the bread - it creates a golden, crispy texture they'll love. Mayo works too and browns more evenly, but butter has a more familiar, comforting taste."},
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 4: With Conversation State/History (Short-term Memory)


For a grilled cheese sandwich with mild flavors that kids will enjoy, I recommend using a combination of mild cheddar and mozzarella. 

- **Mild Cheddar**: This cheese melts well and has a creamy texture with a subtle flavor that won't overpower the sandwich. It melts best at around 150°F (65°C).

- **Mozzarella**: Known for its excellent melting properties, mozzarella adds a gooey texture and a mild, milky flavor. It melts at a slightly lower temperature, around 130°F (54°C), which complements the cheddar perfectly.

Both of these cheeses are readily available in most grocery stores. You can use equal parts of each cheese to create a balanced, kid-friendly grilled cheese. Shred the cheese for faster melting and even distribution. Cook the sandwich on medium-low heat to ensure the cheese melts thoroughly without burning the bread. Enjoy!

In [ ]:
# @title Step 5: Adding Long-term Memory
print("\n" + "="*70)
print("Step 5: With Long-term Memory (Past Sessions)")
print("="*70)

# Information from past conversations (assume stored in a database)
USER_MEMORY = {
    "user_id": "user_123",
    "preferences": {
        "previous_cheese_choice": "mild cheddar (mac and cheese, 2 weeks ago)",
        "dietary_restrictions": "lactose sensitivity",
        "shopping_location": "Kroger",
        "budget_preference": "under $6",
        "household": "cooking for kids"
    }
}

# Load user memory
memory_context = f"""User Profile:
- Previously enjoyed: {USER_MEMORY['preferences']['previous_cheese_choice']}
- Dietary note: {USER_MEMORY['preferences']['dietary_restrictions']}
- Shops at: {USER_MEMORY['preferences']['shopping_location']}
- Budget: {USER_MEMORY['preferences']['budget_preference']}
- Household: {USER_MEMORY['preferences']['household']}"""

# System prompt
system_prompt = f"""You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

{memory_context}

Use this information to personalize your recommendations."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm cooking for my kids who prefer mild flavors"},
    {"role": "assistant", "content": "Great! I'll keep that in mind. Kids often prefer milder, creamier cheeses. What would you like to make?"},
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.2,
)
display(Markdown(response.choices[0].message.content))


Step 5: With Long-term Memory (Past Sessions)


For a kid-friendly grilled cheese sandwich with mild flavors, I recommend using **Havarti** cheese. Here's why:

1. **Mild Flavor**: Havarti has a creamy, buttery taste that is mild enough for kids who might not enjoy stronger flavors. It's similar to the mild cheddar you've enjoyed in the past but with a slightly different profile that can add variety.

2. **Melting Properties**: Havarti melts beautifully, creating a gooey, smooth texture that is perfect for grilled cheese. It melts at a relatively low temperature, around 130°F (54°C), which is ideal for achieving that classic stretchy cheese pull.

3. **Lactose Sensitivity**: Havarti is often lower in lactose compared to other cheeses, making it a better option for those with lactose sensitivity. However, always check the packaging for specific lactose content.

4. **Availability and Budget**: Havarti is commonly available at grocery stores like Kroger, and you should be able to find it within your budget of under $6. Look for store brands or sales to maximize savings.

5. **Practical Cooking Technique**: When making grilled cheese, use medium-low heat to ensure the cheese melts evenly without burning the bread. Butter the outside of the bread for a crispy, golden crust.

If you're looking for an alternative, **Muenster** is another mild cheese with excellent melting properties and a similar price range. It also tends to be lower in lactose.

Enjoy your cooking, and I hope your kids love their grilled cheese sandwiches!

In [ ]:
# @title Step 6: Tool Call Outputs
print("\n" + "="*70)
print("Step 6: With Tool Call Outputs")
print("="*70)

# Simulated tool functions: Results from function calls (APIs, databases, calculations)
check_grocery_inventory = {
    "Kraft American Singles": {"price": 4.99, "in_stock": True},
    "Kroger Mild Cheddar": {"price": 3.49, "in_stock": True},
    "Tillamook Medium Cheddar": {"price": 5.99, "in_stock": True},
    "Boar's Head Vermont Cheddar": {"price": 8.99, "in_stock": True}
}
check_lactose_content = {
    "american": {"lactose_per_slice": "0.1g", "note": "processed, lactose removed"},
    "mild_cheddar": {"lactose_per_slice": "0.2g", "note": "naturally low"},
    "aged_cheddar": {"lactose_per_slice": "0.1g", "note": "aging reduces lactose"}
}
simulate_melting = {
    "american": "Smooth melt, no separation, kid-friendly texture",
    "mild_cheddar": "Good melt, slight oil separation at high heat, richer flavor"
}

# Format tool outputs for context
tool_outputs = f"""Tool Call Results:

Grocery Inventory at Kroger:
{str(check_grocery_inventory)}

Lactose Content Analysis:
{str(check_lactose_content)}

Melting Simulation Results:
{str(simulate_melting)}"""

# System prompt
system_prompt = f"""You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

{memory_context}

Use this information to personalize your recommendations."""

# Inject tool outputs into the prompt
enhanced_user_prompt = f"""{str(tool_outputs)}
Use this real-time data to make your recommendation.
User question: {user_prompt}"""

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": enhanced_user_prompt}
    ],
    temperature=0.2,
)

display(Markdown(response.choices[0].message.content))


Step 6: With Tool Call Outputs


For a delicious and kid-friendly grilled cheese sandwich, I recommend using Kraft American Singles. Here's why:

1. **Lactose Sensitivity**: American cheese is processed in a way that significantly reduces lactose content, making it more suitable for those with lactose sensitivity. With only 0.1g of lactose per slice, it should be gentle on your stomach.

2. **Melting Properties**: American cheese is known for its smooth melt and creamy texture, which is perfect for grilled cheese sandwiches. It melts evenly without separating, ensuring a gooey and satisfying bite that kids will love.

3. **Flavor Profile**: While American cheese is milder than cheddar, it still offers a creamy and slightly tangy flavor that complements the buttery bread of a grilled cheese sandwich.

4. **Availability and Budget**: Kraft American Singles are readily available at Kroger for $4.99, fitting well within your budget of under $6.

5. **Cooking Technique**: For the best results, cook your grilled cheese on medium heat. This allows the cheese to melt thoroughly without burning the bread. Use a non-stick skillet or a griddle, and consider covering the pan with a lid for a minute to help the cheese melt faster.

This choice balances your dietary needs, budget, and the preferences of your household, ensuring a delightful meal for everyone.

In [ ]:
# @title Step 7: Structured Output
print("\n" + "="*70)
print("Step 7: With Structured Output")
print("="*70)

# 🔒 JSON Schema for strict output
schema = {
    "type": "object",
    "properties": {
        "top_recommendation": {"type": "string"},
        "alternatives": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1
        },
        "cooking_tips": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1
        },
        "why_this_works": {"type": "string"}
    },
    "required": ["top_recommendation", "alternatives", "cooking_tips", "why_this_works"],
    "additionalProperties": False
}

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": enhanced_user_prompt}
    ],
    # ✅ Structured Outputs with strict JSON Schema
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "CheeseAdvice",
            "strict": True,          # guarantees valid JSON and schema adherence
            "schema": schema
        }
    },
    temperature=0.2,
)

# The model returns a JSON string in message.content
raw_json = response.choices[0].message.content
data = json.loads(raw_json)

# Pretty print as fenced JSON (or use it programmatically)
display(Markdown(f"```json\n{json.dumps(data, indent=2)}\n```"))


Step 7: With Structured Output


```json
{
  "top_recommendation": "Kraft American Singles",
  "alternatives": [
    "Kroger Mild Cheddar",
    "Tillamook Medium Cheddar"
  ],
  "cooking_tips": [
    "Use medium heat to ensure even melting without burning the bread.",
    "Butter the outside of the bread for a crispy texture.",
    "Cover the pan with a lid to help the cheese melt faster."
  ],
  "why_this_works": "Kraft American Singles are ideal for grilled cheese sandwiches due to their smooth melting properties and low lactose content, making them suitable for lactose-sensitive individuals. They are also budget-friendly and kid-approved for their mild flavor."
}
```

# **Tools, MCP (Model Context Protocol)**

In [ ]:
# Define the LLM model and question
model = "gpt-4o"
user_prompt = "What's the best cheese for a grilled cheese sandwich?"

## **Tools**

Let us build a simple tool that returns random fun facts from Pavlos.

In [ ]:
# --- Fun Facts data ---
PAVLOS_FUN_FACTS = [
    "Mozzarella is the most consumed cheese in the U.S., largely due to pizza.",
    "Gruyère and Emmental are classics for fondue thanks to their smooth melt.",
    "Aging boosts sharpness; 12-month cheddar has nuttier, deeper flavors.",
    "High-moisture cheeses (young gouda, fontina, jack) melt especially evenly.",
    "Rind-on bries are edible; the rind adds mushroomy, earthy notes.",
    "Salt helps control moisture and rind formation during cheesemaking.",
    "Browned spots on grilled cheese = Maillard reaction (not caramelization).",
    "American slices are engineered to melt at lower temperatures.",
    "Fresh cheeses (ricotta, chèvre) soften but don’t stretch like mozzarella.",
    "Taleggio’s washed rind brings savory depth and melts well.",
]

def pavlos_fun_fact():
  return {
      "fact": random.choice(PAVLOS_FUN_FACTS),
      "generated_at": int(time.time())
  }

# ✅ Define a Tool
@function_tool(name_override="pavlos_fun_fact_tool")
async def pavlos_fun_fact_tool() -> dict:
  """Return one random fun fact from Pavlos about cheese. Call multiple times if you want several facts."""
  return pavlos_fun_fact()

# Test tool
print(pavlos_fun_fact())
print(pavlos_fun_fact())
print(pavlos_fun_fact())

{'fact': 'Aging boosts sharpness; 12-month cheddar has nuttier, deeper flavors.', 'generated_at': 1760034511}
{'fact': 'Taleggio’s washed rind brings savory depth and melts well.', 'generated_at': 1760034511}
{'fact': 'Fresh cheeses (ricotta, chèvre) soften but don’t stretch like mozzarella.', 'generated_at': 1760034511}


In [ ]:
# System Prompt
system_prompt = """You are a culinary expert specializing in cheese and comfort foods.
When recommending cheeses, consider:
- Melting properties and optimal temperatures
- Flavor profiles (mild to sharp)
- Availability in typical grocery stores
- Practical cooking techniques

Be specific, practical, and explain your reasoning.

Call the pavlos_fun_fact_tool tool 2–3 times and add facts to your response."""

# Build an agent that uses "pavlos_fun_fact_tool"
culinary_expert_agent = Agent(
    name="Culinary Expert",
    instructions=system_prompt,
    model=model,
    tools=[pavlos_fun_fact_tool],
    model_settings=ModelSettings(temperature=0.2),
)

print("\n" + "="*50)
print("Culinary Expert Agent (with fun facts tool call):",model)
print("="*50)
agent_result = await Runner.run(culinary_expert_agent, input=user_prompt)
print(f"Final Response:")
display(Markdown(agent_result.final_output))


Culinary Expert Agent (with fun facts tool call): gpt-4o
Final Response:


For a classic grilled cheese sandwich, the best cheese is one that melts well and offers a rich, creamy flavor. Here are some top choices:

1. **American Cheese**: 
   - **Melting Properties**: American cheese slices are engineered to melt at lower temperatures, making them perfect for a gooey grilled cheese.
   - **Flavor Profile**: Mild and creamy, with a hint of tang.
   - **Availability**: Widely available in most grocery stores.
   - **Cooking Tip**: Use medium heat to ensure even melting without burning the bread.

2. **Cheddar Cheese**:
   - **Melting Properties**: Sharp cheddar melts well, especially when grated.
   - **Flavor Profile**: Offers a sharper, more robust flavor compared to American cheese.
   - **Availability**: Readily available in mild, medium, and sharp varieties.
   - **Cooking Tip**: Combine with a milder cheese like mozzarella for a balanced flavor and texture.

3. **Mozzarella**:
   - **Melting Properties**: Known for its excellent melting quality, mozzarella adds a stretchy texture.
   - **Flavor Profile**: Mild and milky, it complements stronger cheeses.
   - **Availability**: Extremely popular and easy to find, especially in shredded form.
   - **Cooking Tip**: Pair with a sharper cheese like cheddar for added depth.

4. **Brie**:
   - **Melting Properties**: Brie melts into a creamy, luscious texture.
   - **Flavor Profile**: Offers a buttery, slightly earthy taste due to its edible rind.
   - **Availability**: Available in most cheese sections; look for rind-on varieties for added flavor.
   - **Cooking Tip**: Slice thinly and pair with a sweet element like apple slices for a gourmet twist.

### Fun Cheese Facts:
- Rind-on bries are edible; the rind adds mushroomy, earthy notes.
- American slices are engineered to melt at lower temperatures.
- Mozzarella is the most consumed cheese in the U.S., largely due to pizza.

These cheeses can be mixed and matched to create your perfect grilled cheese sandwich, balancing meltability and flavor to suit your taste. Enjoy experimenting!

In [ ]:
# If you want to list all the tools called in the agent run
# Tools Called
tool_calls = [it for it in agent_result.new_items if isinstance(it, ToolCallItem)]
print("Tools called:")
for i, it in enumerate(tool_calls, 1):
    raw = it.raw_item
    name = getattr(raw, "name", getattr(raw, "tool_name", "unknown_tool"))
    args = getattr(raw, "arguments", getattr(raw, "input", None))
    print(f"{i}. {name}  args={args}")

# Tools Output
tool_outputs = [it for it in agent_result.new_items if isinstance(it, ToolCallOutputItem)]
print("\nTool outputs:")
for i, it in enumerate(tool_outputs, 1):
    raw = it.raw_item
    print(f"{i}. {it.output!r}")

Tools called:
1. pavlos_fun_fact_tool  args={}
2. pavlos_fun_fact_tool  args={}
3. pavlos_fun_fact_tool  args={}

Tool outputs:
1. {'fact': 'Rind-on bries are edible; the rind adds mushroomy, earthy notes.', 'generated_at': 1760034545}
2. {'fact': 'American slices are engineered to melt at lower temperatures.', 'generated_at': 1760034545}
3. {'fact': 'Mozzarella is the most consumed cheese in the U.S., largely due to pizza.', 'generated_at': 1760034545}


## **MCP (Model Context Protocol)**

## **Using Tools from MCP Server**

In a real use case we would not have to build the tool ourselves. In our examples let's say Pavlos mantins his cheese fun facts in a PostrgresSQL database. Tradionally if we need to integrate with the fun facts database. He would give us access to the database or create some API for us.

Like we saw in lecture the limitation of this approach is we have to built custom integrations for every tool. This is why we use the concept of MCP. Pavlos will build an MCP Server and give us the link of the server.

In [ ]:
# MCP Server
pavlos_mcp_url = "https://pavlos-fun-facts.dlops.io/mcp"

async def run_agent(query: str):
  async with AsyncExitStack() as stack:
      # Build a list of MCP servers we want to use
      mcp_servers = []

      # Connect to the Pavlos Cheese MCP server for this session
      mcp_server = await stack.enter_async_context(
          MCPServerStreamableHttp(
              name="Pavlos FastMCP",
              params={"url": pavlos_mcp_url},          # MCP endpoint
              cache_tools_list=True,                   # optional: cache tool discovery
              max_retry_attempts=3,
              client_session_timeout_seconds=180,
          )
      )
      mcp_servers.append(mcp_server)

      # Agent that uses MCP for tool use
      agent = Agent(
          name="Culinary Expert (MCP)",
          instructions=system_prompt,
          model=model,
          # IMPORTANT: add the server in mcp_servers (not tools)
          mcp_servers=mcp_servers,
          model_settings=ModelSettings(temperature=0.2, tool_choice="auto"),
      )
      result = await Runner.run(agent, input=query)
      return result.final_output

# Call it:
out = await run_agent(user_prompt)
from IPython.display import Markdown, display
display(Markdown(out))

For the perfect grilled cheese sandwich, consider using a combination of cheeses to achieve the ideal balance of flavor and meltability. Here are some top choices:

1. **Cheddar**: A classic choice, cheddar offers a sharp, tangy flavor that pairs well with the buttery bread. Opt for medium to sharp cheddar for the best melt. It melts at around 150°F (65°C), making it perfect for a grilled cheese.

2. **American Cheese**: Known for its smooth melting properties, American cheese creates a creamy texture. It's widely available and melts evenly, making it a staple in many grilled cheese recipes.

3. **Gruyère**: This Swiss cheese adds a nutty, slightly sweet flavor. It melts beautifully, making it an excellent choice for a more gourmet grilled cheese.

4. **Fontina**: With a mild, buttery flavor, Fontina melts smoothly and adds a rich creaminess to your sandwich.

5. **Taleggio**: This cheese has a washed rind that brings savory depth and melts well, adding complexity to your grilled cheese.

### Tips for the Perfect Grilled Cheese:
- **Use a combination**: Mixing cheeses like cheddar and Gruyère can give you both sharpness and creaminess.
- **Low and slow**: Cook your sandwich on medium-low heat to ensure the cheese melts before the bread burns.
- **Butter the bread**: Spread butter on the outside of the bread for a golden, crispy crust.

### Fun Cheese Facts:
- Taleggio’s washed rind brings savory depth and melts well.
- Fresh cheeses like ricotta and chèvre soften but don’t stretch like mozzarella.
- Salt helps control moisture and rind formation during cheesemaking.

These insights will help you craft a delicious and satisfying grilled cheese sandwich!

## **Build your own MCP Server** (Optional)

This section is optional where we can build and host our own MCP. Here we use the python framework FastMCP to build an MCP server.

In [ ]:
# --- Define your MCP server with one tool: pavlos_fun_fact
mcp = FastMCP("Pavlos Cheese MCP")

# ✅ Define a Tool as MCP Server
@mcp.tool
def pavlos_fun_fact_mcp() -> dict:
    """Return one random fun fact from Pavlos about cheese. Call multiple times if you want several facts."""
    return pavlos_fun_fact()

# 🏃‍♀️ Run the server over HTTP on port 8000 in a background thread
def run_server():
    # HTTP transport serves the MCP endpoint at /mcp
    mcp.run(transport="http", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("FastMCP server starting on http://127.0.0.1:8000/mcp")

FastMCP server starting on http://127.0.0.1:8000/mcp




╭────────────────────────────────────────────────────────────────────────────╮
│                                                                            │
│        _ __ ___  _____           __  __  _____________    ____    ____     │
│       _ __ ___ .'____/___ ______/ /_/  |/  / ____/ __ \  |___ \  / __ \    │
│      _ __ ___ / /_  / __ `/ ___/ __/ /|_/ / /   / /_/ /  ___/ / / / / /    │
│     _ __ ___ / __/ / /_/ (__  ) /_/ /  / / /___/ ____/  /  __/_/ /_/ /     │
│    _ __ ___ /_/    \____/____/\__/_/  /_/\____/_/      /_____(*)____/      │
│                                                                            │
│                                                                            │
│                                FastMCP  2.0                                │
│                                                                            │
│                                                                            │
│                🖥️  Server name:     Pavlos Chees

Next we will use this local MCP server as the source of our Tool.

In [ ]:
# MCP Server
pavlos_mcp_url = "http://127.0.0.1:8000/mcp"

async def run_agent(query: str):
  async with AsyncExitStack() as stack:
      # Build a list of MCP servers we want to use
      mcp_servers = []

      # Connect to the Pavlos Cheese MCP server for this session
      mcp_server = await stack.enter_async_context(
          MCPServerStreamableHttp(
              name="Pavlos FastMCP",
              params={"url": pavlos_mcp_url},
              cache_tools_list=True,
              max_retry_attempts=3,
          )
      )
      mcp_servers.append(mcp_server)

      # Agent that uses MCP for tool use
      agent = Agent(
          name="Culinary Expert (MCP)",
          instructions=system_prompt,
          model=model,
          # IMPORTANT: add the server in mcp_servers (not tools)
          mcp_servers=mcp_servers,
          model_settings=ModelSettings(temperature=0.2, tool_choice="auto"),
      )
      result = await Runner.run(agent, input=query)
      return result.final_output

# Call it:
out = await run_agent(user_prompt)
from IPython.display import Markdown, display
display(Markdown(out))

INFO:     127.0.0.1:42808 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42810 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:42818 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42824 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42838 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42846 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42862 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35734 - "DELETE /mcp HTTP/1.1" 200 OK


For the perfect grilled cheese sandwich, consider using a combination of cheeses to achieve the ideal melt and flavor. Here are some top choices:

1. **Cheddar**: A classic choice, cheddar offers a sharp, tangy flavor that pairs well with the buttery bread. It melts beautifully, especially when aged for a medium period. Opt for a medium or sharp cheddar for the best balance of flavor and meltability.

2. **Gruyère**: Known for its smooth melting properties, Gruyère adds a nutty and slightly sweet flavor. It's a classic choice for fondue, making it perfect for a grilled cheese that requires a gooey texture.

3. **Fontina**: This cheese is high in moisture, allowing it to melt evenly. It has a mild, buttery flavor that complements stronger cheeses like cheddar or Gruyère.

4. **Monterey Jack**: With its mild flavor and excellent melting properties, Monterey Jack is a great addition to a grilled cheese blend. It helps achieve that stretchy, gooey texture.

### Practical Tips:
- **Temperature**: Cook your sandwich on medium-low heat to ensure the cheese melts thoroughly without burning the bread.
- **Combination**: Try a mix of cheddar and Gruyère or cheddar and Monterey Jack for a balanced flavor and texture.
- **Availability**: All these cheeses are typically available in most grocery stores, making them convenient choices.

### Fun Cheese Facts:
- Fresh cheeses like ricotta and chèvre soften but don’t stretch like mozzarella.
- Gruyère and Emmental are classics for fondue thanks to their smooth melt.
- High-moisture cheeses such as young gouda, fontina, and jack melt especially evenly.

These insights should help you craft a delicious and satisfying grilled cheese sandwich!

# **Multi Agent System: The Cheese Quiz Guild 🧀**

We will build the Cheese Quiz Guild 🧀 with the following requirements:

*   A user can request a quiz on any specific cheese topic  (e.g., "French Cheeses," "Hard Cheeses," "Cheeses from Star Wars")
*   The Guild of AI agents must work together to research, write, verify, and format a high-quality quiz.
* This implementation will illustrate the core principles of AI agents: specialized roles, tool usage, and collaborative workflows.


First we will define some utilities required to build The Cheese Quiz Guild Agents.

In [ ]:
# @title Some Utils

SCHEMA_PATH = Path("quiz-schema.json")

with open("quiz-ui.html", "r") as f:
    QUIZ_UI = f.read()

class QuizAgentOutputSchema(AgentOutputSchemaBase):
    """
    Custom output schema that:
      - returns your JSON Schema (for Structured Outputs)
      - validates and parses the model's JSON into a Python object
    """

    def __init__(self, schema_path: Path = SCHEMA_PATH):
        self._schema_path = schema_path
        self._schema: Dict[str, Any] = json.loads(
            self._schema_path.read_text(encoding="utf-8")
        )
        # tip: assert top-level shape matches Structured Outputs support
        # (only supported/strict JSON Schema features) per docs:
        # https://platform.openai.com/docs/guides/structured-outputs/supported-schemas
        # The uploaded file uses only supported constructs (object/array/string/integer). :contentReference[oaicite:1]{index=1}

    # --- AgentOutputSchemaBase methods required by the SDK ---
    def is_plain_text(self) -> bool:
        return False

    def name(self) -> str:
        return "QuizJSON"

    def json_schema(self) -> Dict[str, Any]:
        # The SDK will pass this to the model as the response schema. :contentReference[oaicite:2]{index=2}
        return self._schema

    def is_strict_json_schema(self) -> bool:
        # Opt into strict mode so the SDK enforces the subset supported by Structured Outputs. :contentReference[oaicite:3]{index=3}
        return True

    def validate_json(self, json_str: str) -> Any:
        """
        Must return the validated object or raise ModelBehaviorError.
        """
        try:
            obj = json.loads(json_str)
        except Exception as e:
            raise ModelBehaviorError(f"Invalid JSON: {e}") from e

        try:
            js_validate(instance=obj, schema=self._schema)  # runtime guardrail
        except ValidationError as e:
            # Signal to the SDK that the model misbehaved. :contentReference[oaicite:4]{index=4}
            raise ModelBehaviorError(
                f"JSON failed schema validation: {e.message}"
            ) from e

        return obj


def render_quiz_html(
    quiz: Dict[str, Any],
    *,
    title: str | None = None,
    shuffle_questions: bool = False,
    shuffle_options: bool = False,
) -> str:

    topic = title or quiz.get("topic") or "Quiz"

    norm_questions = []
    for idx, q in enumerate(quiz.get("questions", []), start=1):
        # Try both shapes
        q_text = q.get("question") or q.get("text") or ""
        opts = q.get("choices") or q.get("options") or []
        # Keys/letters
        normalized_opts = []
        for opt in opts:
            key = opt.get("key") or opt.get("letter") or ""
            normalized_opts.append({"key": str(key), "text": opt.get("text", "")})
        answer_key = q.get("answer_key") or q.get("answer") or ""
        norm_questions.append(
            {
                "id": q.get("id") or idx,
                "text": q_text,
                "options": normalized_opts,
                "answer": str(answer_key),
            }
        )

    client_payload = {
        "title": topic,
        "questions": norm_questions,
        "settings": {
            "shuffleQuestions": bool(shuffle_questions),
            "shuffleOptions": bool(shuffle_options),
        },
    }

    # Escape + serialize once for safe embedding in a <script> tag
    data_json = json.dumps(client_payload, ensure_ascii=False)

    return QUIZ_UI.replace("{ data_json }", data_json)


# ───────────────────────────
# Shared types
# ───────────────────────────
class CheeseFact(BaseModel):
    statement: str
    source: Optional[str] = None

class QuizChoice(BaseModel):
    key: str
    text: str


class QuizQ(BaseModel):
    question: str
    choices: List[QuizChoice]
    answer_key: str
    source_note: Optional[str] = None


class QuizDraft(BaseModel):
    topic: str
    questions: List[QuizQ]


class ReviewResult(BaseModel):
    approved: bool
    notes: str = ""
    needs_revision_indices: List[int] = Field(default_factory=list)
    needs_user_opinion: bool = False


@dataclass
class GuildContext:
    fact_cache: dict[str, List[CheeseFact]]


@dataclass
class OrchestratorResult:
    response_text: str
    structured_json: dict
    formatted_md: str


# ───────────────────────────
# Helper tools
# ───────────────────────────


@function_tool
def ensure_unique_answers(question: str, choices: List[str]) -> bool:
    norm = [c.strip().lower() for c in choices]
    return len(set(norm)) == len(norm)


@function_tool
def ask_head_cheesemonger(prompt: str, default_decision: bool = True) -> bool:
    # In real UI, ask the user. Here we simulate.
    return default_decision

## **Meet the Agents: The Guild Members**

In this section we build the core agents required for our multi agent system. Giving our agents distinct personas will make them more specialized in their roles.

#### 🧐 **Professor Stilton, the Fact-Finder**
This agent's primary job is to take the user's topic and use a **search tool** to find interesting, verifiable facts.

#### ✍️ **Brie, the Creative Wordsmith**
The Brie Agent takes the raw facts from Professor Stilton and transforms them into engaging quiz questions (multiple choice, true/false, etc.). She's the core "creative" LLM.

#### 깐 **Cheddar, the Sharp Critic**
Cheddar Agent reviews the questions and answers created by Brie. He checks them against Stilton's original facts for accuracy. He also ensures the questions are clear and not too easy or impossibly hard.

#### 🎨 **Mozzarella, the Master Formatter**
The Mozzarella Agent takes a quiz that is approved by Cheddar and converts to markdown format


#### 🎨 **Ricotta, the Master Formatter**
Ricotta takes a quiz that is approved by Cheddar and converts into strict JSON

In [ ]:
# Professor Stilton, the Fact-Finder
stilton_agent = Agent(
    name="Professor Stilton, the Fact-Finder",
    model="gpt-4o",
    handoff_description="Gathers concise, citable cheese facts for a topic.",
    instructions=(
        "You are Professor Stilton. Given a cheese topic, use web search to gather ~5 concise, "
        "verifiable facts suitable for quiz questions. Include short source notes (domain or title). "
    ),
    tools=[WebSearchTool()],
    output_type=List[CheeseFact],
    model_settings=ModelSettings(temperature=0.3, tool_choice="auto"),
)

# Test
result = await Runner.run(stilton_agent, "Cheese topic: brie")
cheese_facts = result.final_output
for fact in cheese_facts:
    print(f"{fact.statement} ({fact.source})")

Brie is a soft-ripened cow’s‑milk cheese named after the Brie region in northern France (modern Seine‑et‑Marne), and its name derives from the Gaulish word “briga,” meaning “hill” or “height.” (Wikipedia (Brie) ([en.wikipedia.org](https://en.wikipedia.org/wiki/Brie?utm_source=openai)))
Authentic regional varieties like Brie de Meaux and Brie de Melun have Appellation d’Origine Contrôlée (AOC) protection since 1980, and are traditionally made from unpasteurized milk. (Wikipedia (Brie) ([en.wikipedia.org](https://en.wikipedia.org/wiki/Brie?utm_source=openai)))
Brie is made by molding soft curds into flat rounds (9–15 inches in diameter), then spraying them with Penicillium candidum mold to form an edible white rind; it ripens in about 3–4 weeks. (Britannica ([britannica.com](https://www.britannica.com/topic/Brie-cheese?utm_source=openai)))
A 1‑ounce (28 g) serving of full‑fat Brie provides approximately 95–100 calories, about 4–6 g of protein, 8–9 g of fat (including ~5 g saturated fat),

In [ ]:
# Brie, the Creative Wordsmith
brie_agent = Agent(
    name="Brie, the Creative Wordsmith",
    model="gpt-4o",
    handoff_description="Writes engaging multiple-choice questions from facts.",
    instructions=(
        "Turn the given facts into exactly 5 multiple-choice questions. Each has 4 options (keys a,b,c,d) "
        "with one correct answer. Make them fun but precise; avoid trickiness; include source_note if present. "
        "Return ONLY structured QuizDraft."
    ),
    output_type=QuizDraft,
    model_settings=ModelSettings(temperature=0.7),
)

# Test
input = "Facts:\n" + "\n".join(f"- {f.statement}" for f in cheese_facts)
result = await Runner.run(brie_agent, input=input)
quiz_draft = result.final_output
display(quiz_draft)

QuizDraft(topic='Brie Cheese', questions=[QuizQ(question="What is the origin of the name 'Brie'?", choices=[QuizChoice(key='a', text="It comes from the Italian word for 'cheese'."), QuizChoice(key='b', text='It is named after a famous French king.'), QuizChoice(key='c', text="It derives from the Gaulish word 'briga' meaning 'hill' or 'height'."), QuizChoice(key='d', text='It is a modern name given in the 20th century.')], answer_key='c', source_note=None), QuizQ(question='Which of the following is true about Brie de Meaux and Brie de Melun?', choices=[QuizChoice(key='a', text="They are made from goat's milk."), QuizChoice(key='b', text='They have Appellation d’Origine Contrôlée (AOC) protection.'), QuizChoice(key='c', text='They are always made from pasteurized milk.'), QuizChoice(key='d', text='They are the only types of Brie available worldwide.')], answer_key='b', source_note=None), QuizQ(question='How is the white rind on Brie cheese formed?', choices=[QuizChoice(key='a', text='By 

In [ ]:
# Cheddar, the Sharp Critic
cheddar_agent = Agent(
    name="Cheddar, the Sharp Critic",
    handoff_description="Reviews quiz clarity & correctness vs facts; requests fixes.",
    instructions=(
        "Compare each question to the supplied facts. Flag ambiguous, too-easy/too-hard, or ungrounded items. "
        "Use ensure_unique_answers to detect duplicate options. If borderline difficulty or taste-based, "
        "set needs_user_opinion=True. Produce ReviewResult with indices & notes."
    ),
    tools=[ensure_unique_answers, ask_head_cheesemonger],
    output_type=ReviewResult,
    model_settings=ModelSettings(temperature=0.2),
)

# Mozzarella, the Master Formatter
mozzarella_agent = Agent(
    name="Mozzarella, the Master Formatter",
    handoff_description="Formats the approved quiz for delivery.",
    instructions=(
        "Format the final, approved QuizDraft into Markdown:\n"
        "• Title\n• Numbered questions\n• Lettered options (a–d)\n• **Answer Key**\n• Optional sources."
    ),
    output_type=str,
    model_settings=ModelSettings(temperature=0.2),
)

# Cashew Agent, Dynamic specialist
cashew_agent = Agent(
    name="Cashew, the Plant-Based Pro",
    handoff_description="Adds vegan/plant-based cheese expertise to research.",
    instructions=(
        "Augment Stilton with up to 3 extra facts when the topic includes vegan/plant-based."
    ),
    tools=[WebSearchTool()],
    output_type=List[CheeseFact],
    model_settings=ModelSettings(temperature=0.4),
)

# Ricotta Agent
ricotta_agent = Agent(
    name="Ricotta, the Schema Scribe",
    handoff_description="Converts an approved quiz into strict JSON that matches the provided schema.",
    instructions=(
        "You will receive an approved quiz (questions with options and the correct answer). "
        "Output ONLY valid JSON that conforms to the provided schema. No markdown, no comments.\n"
        "RULES:\n"
        "1) Output ONLY raw JSON (no Markdown fences, no commentary).\n"
        "2) Use sequential integer IDs starting at 1.\n"
        "3) Each question MUST have exactly four options with 'letter' keys: 'a', 'b', 'c', 'd'.\n"
        "4) The 'answer' must equal one of the option letters ('a'|'b'|'c'|'d').\n"
        "5) Preserve the original question wording and option texts; do not invent facts.\n"
        "6) Do not include sources or extra fields—only the fields in the schema.\n"
    ),
    output_type=QuizAgentOutputSchema(),  # <- custom schema object
    model_settings=ModelSettings(temperature=0.1),
)

## **Define Tools**

In this section we will create Agents-as-tools for orchestration

In [ ]:
# Agents-as-tools for orchestration
stilton_tool = stilton_agent.as_tool(
    "research_facts", "Find ~5 concise, citable facts for a topic."
)
brie_tool = brie_agent.as_tool("write_quiz", "Convert facts into a 5-question MCQ quiz.")
cheddar_tool = cheddar_agent.as_tool("review_quiz", "QA the quiz against the facts.")
mozzarella_tool = mozzarella_agent.as_tool("format_quiz", "Format quiz into Markdown.")
ricotta_tool = ricotta_agent.as_tool("format_quiz_json", "Format quiz into JSON.")
cashew_tool = cashew_agent.as_tool(
    "vegan_specialist", "Supplement with vegan cheese facts as needed."
)

## **Agent Orchestrator**

In this section we build the workflow that assembles all the agents together.

1.  **The Order (Input)**: The user requests a quiz on "Italian Cheeses."
2.  **Research (Agent 1)**: The request goes to **Professor Stilton**. He searches for 5 key facts about Pecorino, Parmesan, Mozzarella, etc.
3.  **Creation (Agent 2)**: The facts are passed to **Brie**. She crafts 5 multiple-choice questions based on the facts.
4.  **Verification (Agent 3)**: The draft quiz is sent to **Cheddar**. He flags one question as potentially confusing and sends it back to Brie with a note. Brie revises it and sends it back to Cheddar, who now approves the whole set.
5.  **Formatting (Agent 4)**: The final, approved quiz goes to **Ricotta**, who formats it into structured JSON
6.  **Delivery (Output)**: The final, polished quiz is presented to the user.


In [ ]:
orchestrator = Agent(
    name="Cheese Quiz Guild Orchestrator",
    instructions=(
        "Act as the head of the Cheese Quiz Guild. "
        "Follow the assembly line strictly:\n"
        "1) Call research_facts(topic). If the topic is vegan/plant-based, also call vegan_specialist.\n"
        "   Merge/trim to ~5 total facts (no duplicates), and update context.fact_cache[topic].\n"
        "2) Call write_quiz(topic, facts).\n"
        "3) Call review_quiz(draft, facts). If not approved and not needs_user_opinion, "
        "   send exactly one revision request back to write_quiz and then re-review. "
        "   (Make at most 2 total review rounds.)\n"
        "   If needs_user_opinion, call ask_head_cheesemonger to decide and proceed.\n"
        "4) format_quiz_json(approved_draft) → JSON.\n"
        "Keep outputs structured between steps; only the last step returns JSON."
    ),
    tools=[
        stilton_tool,
        cashew_tool,
        brie_tool,
        cheddar_tool,
        ricotta_tool,
        ask_head_cheesemonger,
    ],
    output_type=QuizAgentOutputSchema(),
    model_settings=ModelSettings(temperature=0.3),
)

In [ ]:
async def run_guild(topic: str) -> str:

    # seed the orchestrator with the user's topic
    prompt = f"Customer requests a quiz on: {topic}. Produce the final JSON."
    result = await Runner.run(
        orchestrator, prompt, max_turns=12
    )
    quiz = result.final_output
    html_str = render_quiz_html(quiz, shuffle_questions=False, shuffle_options=False)
    return HTML(html_str)

# **Cheese Quiz Guild Agent in Action**

## Example 1:

In [ ]:
# Create a quiz
await run_guild("Italian Cheeses")

## Example 2:

In [ ]:
# Create a quiz
await run_guild("Greek cheeses")